In [1]:
import numpy as np
import numpy.typing as npt
from astropy.nddata import block_replicate, block_reduce

In [ ]:
def upscale(
    data: npt.NDArray,
    upscale_y: int = 1,
    upscale_x: int = 1,
) -> npt.NDArray:
    """
    Upscale a 2D array by repeating elements along each axis.

    Args:
        data (npt.NDArray): Input 2D array.
        upscale_y (int): Upscaling factor over the y direction.
        upscale_x (int): Upscaling factor over the x direction.

    Returns:
        output (npt.NDArray): Oversampled array.

    Raises:
        ValueError: if upscale factors are not positive integers.
    
    Notes:
        - The array total sum is conserved through linear interpolation.
        - For N-dim arrays, consider using `astropy.nndata.block_replicate()`.
    """
    if not (
        (isinstance(upscale_y, int) and upscale_y > 0) and
        (isinstance(upscale_x, int) and upscale_x > 0)
    ):
        raise ValueError("Upscaling factors must be positive integers.")
    
    for i, f in enumerate((upscale_y, upscale_x)):
        data = np.repeat(data, f, axis=i)
    
    return data/np.prod((upscale_y, upscale_x))


def downscale(
    data: npt.NDArray,
    downscale_y: int = 1,
    downscale_x: int = 1,
) -> npt.NDArray:
    """
    Downscale a 2D array.

    Args:
        data (npt.NDArray): Input 2D array.
        downscale_y (int): Downscaling factor over the y direction.
        downscale_x (int): Downscaling factor over the x direction.

    Returns:
        output (npt.NDArray): Downsampled array.

    Raises:
        ValueError: if downscale factors are not positive integers.
    
    Notes:
        - The downsampling is performed through blocks subdivision, which
          represent the elements of the downsampled array. Each block is
          reduced by adding its elements for linear interpolation.
        - The total sum of the array is conserved.
        - For N-dim arrays, consider using `astropy.nndata.block_reduce()`.
    """
    def _handle_shape(
        data: npt.NDArray,
        downscaling: npt.NDArray,
    ) -> npt.NDArray:
        """Adjusts array for blocks subdivision by cutting extra-rows/columns."""
        def _handle_axis(a: npt.NDArray, idx: int) -> npt.NDArray:
            """Redistributes cutted values in the block-adjusted axis."""
            return a[:idx] + a[idx:].sum(axis=0) / idx
        adj_shape = (np.array(data.shape) // downscaling) * downscaling
        for ax in range(data.ndim):
            if data.shape[ax] != adj_shape[ax]:
                data = data.swapaxes(0, ax)
                data = _handle_axis(data, adj_shape[ax])
                data = data.swapaxes(0, ax)
        return data

    def _to_blocks(
        data: npt.NDArray,
        downscaling: npt.NDArray,
    ) -> npt.NDArray:
        """Reshapes input array into blocks."""
        assert not np.any(np.mod(data.shape, downscaling) != 0)
        nblocks = np.array(data.shape) // downscaling
        reshaping = tuple(dim for dims in zip(nblocks, downscaling) for dim in dims)
        return data.reshape(reshaping).transpose((0, 2, 1, 3))
    
    if not (
        (isinstance(downscale_y, int) and downscale_y > 0) and
        (isinstance(downscale_x, int) and downscale_x > 0)
    ):
        raise ValueError("Downscaling factors must be positive integers.")

    downscaling = np.array((downscale_y, downscale_x))
    data = _handle_shape(data, downscaling)
    data = _to_blocks(data, downscaling)
    return data.sum(axis=(2, 3))

In [3]:
import mbloodmoon as bm
from IROS_pipeline import _handle_dirpaths

mask_FITS = "wfm_mask.fits"

skyfield = "GalacticCenter"
data_FITS = "20241011_galctr_rxte_sax_2-30keV_1ks_2cams_sources_cxb"

mask_file, simul_data, save_path = _handle_dirpaths(
    mask=mask_FITS,
    skyfield=skyfield,
    simul=data_FITS,
)

wfm = bm.codedmask(mask_file, upscale_x=1, upscale_y=1)

In [6]:
sky = np.ones(wfm.sky_shape)

down_sky = downscale(sky, *(5, 3))
up_sky = upscale(down_sky, *(5, 3))


sky.shape, up_sky.shape, down_sky.shape, int(sky.sum()), int(up_sky.sum()), int(down_sky.sum())

((1033, 1671), (1030, 1671), (206, 557), 1726143, 1726143, 1726143)

In [15]:
wfm2 = bm.codedmask(mask_file, upscale_x=3, upscale_y=5)

wfm2.sky_shape

(5163, 5015)

In [31]:
def downscale(
    data: np.array,
    downscale_y: int = 1,
    downscale_x: int = 1,
) -> np.array:
    """Downscale a 2D array."""    
    def _handle_shape(
        data: npt.NDArray,
        downscaling: npt.NDArray,
    ) -> npt.NDArray:
        """Adjusts array for blocks subdivision by cutting extra-rows/columns."""
        def _handle_axis(a: npt.NDArray, idx: int) -> npt.NDArray:
            """Redistributes cutted values in the block-adjusted axis."""
            return a[:idx] + a[idx:].sum(axis=0) / idx
        adj_shape = (np.array(data.shape) // downscaling) * downscaling
        for ax in range(data.ndim):
            if data.shape[ax] != adj_shape[ax]:
                data = data.swapaxes(0, ax)
                data = _handle_axis(data, adj_shape[ax])
                data = data.swapaxes(0, ax)
        return data

    def _to_blocks(
        data: npt.NDArray,
        downscaling: npt.NDArray,
    ) -> npt.NDArray:
        """Reshapes input array into blocks."""
        assert not np.any(np.mod(data.shape, downscaling) != 0)
        nblocks = np.array(data.shape) // downscaling
        reshaping = tuple(dim for dims in zip(nblocks, downscaling) for dim in dims)
        return data.reshape(reshaping).transpose((0, 2, 1, 3))
    
    if not (
        (isinstance(downscale_y, int) and downscale_y > 0) and
        (isinstance(downscale_x, int) and downscale_x > 0)
    ):
        raise ValueError("Downscaling factors must be positive integers.")

    downscaling = np.array((downscale_y, downscale_x))
    data = _handle_shape(data, downscaling)
    data = _to_blocks(data, downscaling)
    return data.sum(axis=(2, 3))






a = np.ones(100).reshape((10, 10))
fy, fx = 3, 3
downsampled_a = downscale(a, *(fy, fx))

a, downsampled_a

(array([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]]),
 array([[11.11111111, 11.11111111, 11.11111111],
        [11.11111111, 11.11111111, 11.11111111],
        [11.11111111, 11.11111111, 11.11111111]]))

In [ ]:
aa = a[:9] + a[9:].sum(axis=0)/9
aa = aa[:, :9] + aa[:, 9:].sum(axis=1)/9

In [34]:
n, m = 57, 31
a = np.ones((n, m))

fy, fx = 4, 3
u, v = n // fy, m // fx
b = downscale(a, fy, fx)


aa = a[:u] + a[u:].sum(axis=0) / u                       
aa = aa[:, :v] + aa[:, v:].sum(axis=1, keepdims=True) / v

In [ ]:
a, b


(array([[1., 1., 1., ..., 1., 1., 1.],
        [1., 1., 1., ..., 1., 1., 1.],
        [1., 1., 1., ..., 1., 1., 1.],
        ...,
        [1., 1., 1., ..., 1., 1., 1.],
        [1., 1., 1., ..., 1., 1., 1.],
        [1., 1., 1., ..., 1., 1., 1.]], shape=(57, 31)),
 (14, 10))

In [40]:
aa = a[:u] + a[u:].sum(axis=0) / u                       
aa = aa[:, :v] + aa[:, v:].sum(axis=1, keepdims=True) / v

aa.shape

(14, 10)

In [ ]:
from reproject.utils import parse_input_data

def sky_composition(
    input_data,
    output_projection,
    shape_out,
    reproject_function,
    combine_function,
) -> tuple[np.array, np.array]:
    
    # Parse the output projection to avoid having to do it for each
    wcs_out, shape_out = parse_output_projection(output_projection, shape_out=shape_out)

    output_array = np.zeros(shape_out)

    output_footprint = np.zeros(shape_out)

    on_the_fly = not match_background and combine_function in ("mean", "sum")


    # Start off by reprojecting individual images to the final projection
    if not on_the_fly:
        arrays = []

    with tempfile.TemporaryDirectory(ignore_cleanup_errors=IS_WIN) as local_tmp_dir:
        for idata in range(len(input_data)):
            # We need to pre-parse the data here since we need to figure out how to
            # optimize/minimize the size of each output tile (see below).
            array_in, wcs_in = parse_input_data(input_data[idata], hdu_in=hdu_in)

            # We also get the weights map, if specified
            weights_in = None

            # Since we might be reprojecting small images into a large mosaic we
            # want to make sure that for each image we reproject to an array with
            # minimal footprint. We therefore find the pixel coordinates of the
            # edges of the initial image and transform this to pixel coordinates in
            # the final image to figure out the final WCS and shape to reproject to
            # for each tile. We strike a balance between transforming only the
            # input-image corners, which is fast but can cause clipping in cases of
            # significant distortion (when the edges of the input image become
            # convex in the output projection), and transforming every edge pixel,
            # which provides a lot of redundant information.

            edges = sample_array_edges(array_in.shape, n_samples=11)[::-1]
            edges_out = pixel_to_pixel(wcs_in, wcs_out, *edges)[::-1]

            # Determine the cutout parameters

            # In some cases, images might not have valid coordinates in the corners,
            # such as all-sky images or full solar disk views. In this case we skip
            # this step and just use the full output WCS for reprojection.

            ndim_out = len(shape_out)
            if np.any(np.isnan(edges_out)):
                bounds = list(zip([0] * ndim_out, shape_out, strict=False))
            else:
                bounds = []
                for idim in range(ndim_out):
                    imin = max(0, int(np.floor(edges_out[idim].min() + 0.5)))
                    imax = min(shape_out[idim], int(np.ceil(edges_out[idim].max() + 0.5)))
                    bounds.append((imin, imax))
                    if imax < imin: break

            slice_out = tuple([slice(imin, imax) for (imin, imax) in bounds])

            if isinstance(wcs_out, WCS):
                wcs_out_indiv = wcs_out[slice_out]
            else:
                wcs_out_indiv = SlicedLowLevelWCS(wcs_out.low_level_wcs, slice_out)

            shape_out_indiv = tuple([imax - imin for (imin, imax) in bounds])

            # TODO: optimize handling of weights by making reprojection functions
            # able to handle weights, and make the footprint become the combined
            # footprint + weight map

            array = footprint = None

            array, footprint = reproject_function(
                (array_in, wcs_in),
                output_projection=wcs_out_indiv,
                shape_out=shape_out_indiv,
                hdu_in=hdu_in,
                output_array=array,
                output_footprint=footprint,
                **kwargs,
            )

            # For the purposes of mosaicking, we mask out NaN values from the array
            # and set the footprint to 0 at these locations.
            reset = np.isnan(array)
            array[reset] = 0.0
            footprint[reset] = 0.0

            array = ReprojectedArraySubset(array, footprint, bounds)

            # TODO: make sure we gracefully handle the case where the
            # output image is empty (due e.g. to no overlap).

            if on_the_fly:
                # By default, values outside of the footprint are set to NaN
                # but we set these to 0 here to avoid getting NaNs in the
                # means/sums.
                array.array[array.footprint == 0] = 0
                output_footprint[array.view_in_original_array] += array.footprint
                # We now need to do output[view] += array * footprint but to avoid
                # the temporary array allocation from array * footprint we modify
                # array inplace, which we can do as the array will be discarded at
                # the end of the loop.
                array.array *= array.footprint
                output_array[array.view_in_original_array] += array.array

            else:
                logger.info(f"Adding reprojected array to list to combine later")
                arrays.append(array)

        if combine_function in ("mean", "sum"):

            if combine_function == "mean":
                with np.errstate(invalid="ignore"):
                    output_array /= output_footprint

        elif combine_function in ("first", "last", "min", "max"):
            if combine_function == "min":
                output_array[...] = np.inf
            elif combine_function == "max":
                output_array[...] = -np.inf

            for array in arrays:
                if combine_function == "first":
                    mask = output_footprint[array.view_in_original_array] == 0
                elif combine_function == "last":
                    mask = array.footprint > 0
                elif combine_function == "min":
                    mask = (array.footprint > 0) & (
                        array.array < output_array[array.view_in_original_array]
                    )
                elif combine_function == "max":
                    mask = (array.footprint > 0) & (
                        array.array > output_array[array.view_in_original_array]
                    )

                output_footprint[array.view_in_original_array] = np.where(
                    mask, array.footprint, output_footprint[array.view_in_original_array]
                )
                output_array[array.view_in_original_array] = np.where(
                    mask, array.array, output_array[array.view_in_original_array]
                )

    # We need to avoid potentially large memory allocation from output == 0 so
    # we operate in chunks.
    logger.info(f"Resetting invalid pixels to {blank_pixel_value}")
    for chunk in iterate_chunks(output_array.shape, max_chunk_size=256 * 1024**2):
        output_array[chunk][output_footprint[chunk] == 0] = blank_pixel_value

    return output_array, output_footprint